# Week 2 — Context Engineering I

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-02-context-engineering-i-content.html`.

**You will practice:**
1. System prompt vs. no system prompt, same question.
2. Zero-shot vs. few-shot prompting on a classification task.
3. Chain-of-thought vs. direct answer on a multi-step problem.
4. Schema-forced JSON output — raw SDK, then a light ADK and LangChain preview.
5. Two open exercises.


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"

## 1. System prompt vs. no system prompt

Same user question, with and without a system instruction.

In [ ]:
question = "My bike's front brake feels loose, what should I check?"

no_system = client.models.generate_content(model=MODEL, contents=question)
print("--- Without system prompt ---")
print(no_system.text)

with_system = client.models.generate_content(
    model=MODEL,
    contents=question,
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a support assistant for a bike rental company. "
            "Only answer questions related to bike rentals, maintenance, and safety. "
            "Keep answers under 3 sentences. If asked something unrelated, politely decline."
        ),
    ),
)
print("\n--- With system prompt ---")
print(with_system.text)

## 2. Zero-shot vs. few-shot

Watch the output format stabilize once examples are added.

In [ ]:
zero_shot = 'Classify the sentiment of this review as positive, negative, or neutral: "It\'s fine, does what it says on the box."'

few_shot = '''Classify the sentiment of a review as positive, negative, or neutral.

Review: "Fast shipping and the product works great."
Sentiment: positive

Review: "It arrived broken and support never replied."
Sentiment: negative

Review: "It's fine, does what it says on the box."
Sentiment:'''

print("Zero-shot:", client.models.generate_content(model=MODEL, contents=zero_shot).text.strip())
print("Few-shot: ", client.models.generate_content(model=MODEL, contents=few_shot).text.strip())

## 3. Chain-of-thought vs. direct answer

In [ ]:
problem = "A store had 120 apples. It sold 35% of them in the morning and 28 more in the afternoon. How many apples are left?"

direct = client.models.generate_content(
    model=MODEL,
    contents=problem + " Answer with just the number.",
    config=types.GenerateContentConfig(temperature=0.1),
)
print("Direct:", direct.text.strip())

cot = client.models.generate_content(
    model=MODEL,
    contents=problem + ' Think step by step, then give the final answer on its own line starting with "Answer:".',
    config=types.GenerateContentConfig(temperature=0.1),
)
print("\nChain-of-thought:\n", cot.text)

## 4. Schema-forced JSON output

### 4a. Raw SDK

In [ ]:
class TicketTriage(BaseModel):
    category: str   # "billing" | "technical" | "account" | "other"
    urgency: str     # "low" | "medium" | "high"
    summary: str

response = client.models.generate_content(
    model=MODEL,
    contents="My card was charged twice for the same order and I need this fixed today.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TicketTriage,
    ),
)
ticket: TicketTriage = response.parsed
print(ticket)
print(ticket.category, "|", ticket.urgency)

### 4b. Google ADK — `output_schema` (light preview)

ADK's `LlmAgent` accepts a Pydantic model directly via `output_schema` and returns validated structured output.

In [ ]:
import asyncio
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

triage_agent = Agent(
    model=MODEL,
    name="triage_agent",
    instruction="Triage the support message into the given schema.",
    output_schema=TicketTriage,
)

def ask_adk_agent(agent, prompt, app_name="week2_app", user_id="student"):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = asyncio.run(runner.session_service.create_session(app_name=app_name, user_id=user_id))
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    for event in runner.run(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

raw_json = ask_adk_agent(triage_agent, "My card was charged twice for the same order and I need this fixed today.")
print(raw_json)
print(TicketTriage.model_validate_json(raw_json))

### 4c. LangChain — `with_structured_output` (light preview)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=MODEL)
structured_llm = llm.with_structured_output(TicketTriage)

result = structured_llm.invoke("My card was charged twice for the same order and I need this fixed today.")
print(result)

## 5. Exercises

In [ ]:
# TODO Exercise 1 — Design a few-shot prompt
# Pick a small labeling task of your own (e.g. classify emails as "spam"/"not spam",
# or tag a sentence with a difficulty level). Write a zero-shot version and a few-shot
# version (3-4 examples) and compare the outputs on 3 new inputs.

# your code here


In [ ]:
# TODO Exercise 2 — Mini "resume line" parser
# Define a Pydantic model `ResumeLine` with fields like `role: str`, `company: str`,
# `years: float`. Use schema-forced JSON output to parse a single free-text line such as:
#   "Senior backend engineer at Northwind Traders for 3.5 years"
# into a ResumeLine instance, and print the parsed object.

class ResumeLine(BaseModel):
    ...  # your fields here

# your generate_content call here


## Next week

Week 3 — **Context Engineering II**: managing the context window (summarization, compression, sliding window)
and an introduction to embeddings and semantic search. See `week-03-context-engineering-ii-content.html`.